# R0. 결과 종합 — 단계별 결정 근거를 그림으로

모든 그림은 실험이 저장한 파일에서 다시 그린다. **표와 그림의 숫자는 모두 실제로 돌려서 잰 값**이다. 그림은 `reports/figures/*.png` 로도 저장되어 발표 자료에 바로 쓴다.

| 단계 | 질문 | 읽는 파일 | 그림 |
|---|---|---|---|
| 1. 학습 문장 수 | 몇 문장으로 학습할까 | `D1_lc_*.json` | 학습량 곡선 |
| 2. 표본 흔들림 | 점수 차이가 우연이 아닌가 | `D2_*.json` | 세트·시드별 점수 분포 |
| 3. 설정 조정 | 에폭·학습률을 얼마로 | `*.json` 의 history | 에폭별 dev 곡선 |
| 4. 모델 비교 | 어떤 모델이 나은가 | `scores/*_test.json` | 점수 막대 + 흔들림 폭 |
| 5. 태그·카테고리 | 어디서 강하고 약한가 | `scores/*` | 열지도 |
| 6. 오답 분석 | 무엇을 틀리나, 서로 보완되나 | `scores/*_dev.json`, `preds/*_dev.jsonl` | 오답 유형 막대, 보완 열지도, 앙상블 표 |

**점수 파일(`scores/`)을 기준으로 모으는 이유**: 우리 학습 코드로 돌린 실험이든, 팀원이 자기 코드로 만든 예측을 `src/evaluate.py` 로 채점한 것이든 같은 형식의 점수 파일이 생긴다. 그래서 모두 한 표·한 그림에 들어간다.

## 0. 공통 준비 (모든 노트북 같음)
Colab → 런타임 → 런타임 유형 변경 → GPU(T4 이상). 구글 드라이브를 연결하면 이 PC의 프로젝트 폴더가 `/content/drive/Othercomputers/내 PC/...` 아래에 보인다. 경로가 다르면 왼쪽 파일 탭에서 찾아 `PROJECT` 만 고친다.

공용 코드 `src/ner_lib.py`(학습)와 `src/evaluate.py`(팀 공용 채점기)를 불러온다. 정렬·복원·채점 규칙이 모든 노트북에서 같아야 비교가 공정하기 때문이다.

**저장되는 것**: 실험마다 `reports/results/{실험}.json`(에폭별 dev 점수 포함), `preds/`(예측), `scores/`(채점), `errors/`(오답), 그리고 `runs.jsonl`(실행 기록 한 줄). 그림은 R0 노트북이 이 파일들로 그린다.
**끊겼을 때**: 같은 셀을 다시 실행하면 끝난 실험은 건너뛰고, 진행 중이던 실험은 마지막 에폭 체크포인트부터 이어서 학습한다.

In [ ]:
!pip -q install -U transformers datasets accelerate pytorch-crf kiwipiepy
import sys, os, json, torch
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/Othercomputers/내 PC/NLP/프로젝트1/공통2_개체명인식_NER"
sys.path.insert(0, f"{PROJECT}/src")
import importlib, ner_lib; importlib.reload(ner_lib)
MODELS = "/content/drive/MyDrive/ner_models"   # 최종 가중치(실험마다 폴더). 학습 직후 저장하고 다시 읽어 확인한다. 지우지 말 것
ner_lib.DEFAULTS.update(model_dir=MODELS)
from ner_lib import run_experiment, load_results, results_table, upper_bound, load_jsonl, prepare_aihub_model
SPLIT   = f"{PROJECT}/data/split_v1"          # dev 1만 / test 1만 (모든 버전 공통)
TRAIN   = f"{PROJECT}/data/split_v3/train.jsonl"   # 팀 기준 학습 데이터: 30만 (2026-10-22 결정)
SUFFIX  = "_300k"                              # 30만으로 학습한 실험 이름 꼬리표
RESULTS = f"{PROJECT}/reports/results"        # 결과 json이 모이는 곳 (팀원 모두 같은 폴더)
CKPT    = "/content/drive/MyDrive/ner_ckpt"    # 에폭 체크포인트. 끊겨도 이어서 학습. '내 드라이브'라 PC로 동기화되지 않음
MEMBER  = "sejin_choi"                          # ← 실행한 사람 이름, 영어로 (결과 기록 runs.jsonl에 남음)
ner_lib.DEFAULTS.update(ckpt_dir=CKPT, member=MEMBER)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

In [ ]:
# 한글 글꼴 (그래프 글자가 네모로 깨지지 않게)
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, glob as _g
for f in _g.glob("/usr/share/fonts/truetype/nanum/*.ttf"): fm.fontManager.addfont(f)
plt.rc("font", family="NanumGothic"); plt.rc("axes", unicode_minus=False)
FIG = f"{PROJECT}/reports/figures"; os.makedirs(FIG, exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=150); plt.show(); print("저장:", f"{FIG}/{name}.png")

In [ ]:
import pandas as pd, numpy as np, statistics as st, glob
import evaluate as ev
res = {r["exp"]: r for r in load_results(RESULTS)}                 # 우리 코드로 학습한 실험 (history·시간 포함)
def load_scores(split):
    out = {}
    for p in glob.glob(f"{RESULTS}/scores/*_{split}.json"):
        s = json.load(open(p, encoding="utf-8")); out[s["name"]] = s
    return out
S_dev, S_test = load_scores("dev"), load_scores("test")
runs = pd.read_json(f"{RESULTS}/runs.jsonl", lines=True) if os.path.exists(f"{RESULTS}/runs.jsonl") else pd.DataFrame()
print("학습 실험", len(res), "| 채점된 결과 dev", len(S_dev), "test", len(S_test))
runs.tail(20)

## 1. 학습 문장 수 결정 (D1)
x축은 로그 스케일. 판단은 **dev 점수**로 한다. 예시 기준: 문장을 3배 이상 늘렸을 때 dev F1이 0.5점 미만 오르면 그 앞 크기로 정한다(기준은 팀이 정해 아래 결론 칸에 적는다).

In [ ]:
lc = sorted([r for k, r in res.items() if k.startswith("D1_lc_")], key=lambda r: r["n_train"])
if lc:
    n = [r["n_train"] for r in lc]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(n, [r["dev"]["ALL"]["f1"] for r in lc], "o-", label="dev F1(micro)")
    ax[0].plot(n, [r["dev"]["MACRO_F1"] for r in lc], "s--", label="dev macro F1")
    for x, r in zip(n, lc): ax[0].annotate(f"{r['dev']['ALL']['f1']:.3f}", (x, r["dev"]["ALL"]["f1"]), fontsize=8, xytext=(0, 6), textcoords="offset points", ha="center")
    ax[0].set_xscale("log"); ax[0].set_xlabel("학습 문장 수"); ax[0].set_ylabel("F1"); ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title("학습량 곡선")
    ax[1].plot(n, [r["train_sec"] / 60 for r in lc], "o-"); ax[1].set_xscale("log"); ax[1].set_yscale("log")
    ax[1].set_xlabel("학습 문장 수"); ax[1].set_ylabel("학습 시간(분)"); ax[1].grid(alpha=.3); ax[1].set_title(f"학습 시간 ({lc[-1]['gpu']})")
    savefig("1_학습량곡선")
    display(pd.DataFrame([{"n_train": r["n_train"], "dev_F1": r["dev"]["ALL"]["f1"], "dev_macro": r["dev"]["MACRO_F1"],
                           **{f"dev_{t}": r["dev"][t]["f1"] for t in ev.TAGS}, "train_min": round(r["train_sec"] / 60, 1)} for r in lc]))

## 2. 표본 흔들림 (D2)
같은 크기의 서로 다른 세트(A·B·C)와 같은 세트의 다른 시드로 잰 test F1이다. 점들이 좁게 모여 있으면 "어떤 표본을 뽑아도 비슷하다" → 고정 세트 하나로 모델을 비교해도 된다는 근거. **std(표준편차)는 아래 모델 비교에서 '의미 있는 차이'의 기준선**으로 쓴다.

In [ ]:
grp = {"표본 A·B·C": ["D2_set_A_seed42", "D2_set_B_seed42", "D2_set_C_seed42"],
       "시드 42·1·2": ["D2_set_A_seed42", "D2_set_A_seed1", "D2_set_A_seed2"]}
NOISE = None
fig, ax = plt.subplots(figsize=(6, 3.5))
for i, (g, names) in enumerate(grp.items()):
    xs = [res[n]["test"]["ALL"]["f1"] for n in names if n in res]
    if not xs: continue
    ax.scatter([i] * len(xs), xs, s=60, zorder=3)
    if len(xs) >= 2:
        m, sd = st.mean(xs), st.stdev(xs); ax.errorbar(i + .15, m, yerr=sd, fmt="D", c="k", capsize=6)
        ax.annotate(f"평균 {m:.4f}\nstd {sd:.4f}", (i + .2, m), fontsize=9)
        NOISE = max(NOISE or 0, sd)
ax.set_xticks(range(len(grp))); ax.set_xticklabels(grp.keys()); ax.set_ylabel("test F1"); ax.grid(alpha=.3); ax.set_title("표본·시드에 따른 흔들림")
savefig("2_표본흔들림")
print("모델 비교에 쓸 흔들림 기준 std =", NOISE)

## 3. 설정 조정 근거 — 에폭별 dev 곡선
실험마다 매 에폭 끝에 잰 dev F1이다. 곡선이 평평해지거나 내려가기 시작하는 에폭이 적정 에폭이다. 학습률을 바꿔 본 실험(`_lr` 이 붙은 것)도 함께 나온다.

In [ ]:
hs = {k: r["history"] for k, r in res.items() if r.get("history")}
stages = sorted({res[k]["config"].get("stage", "") for k in hs})
fig, axes = plt.subplots(1, max(1, len(stages)), figsize=(5 * max(1, len(stages)), 3.8), squeeze=False)
for ax, stg in zip(axes[0], stages):
    for k, h in hs.items():
        if res[k]["config"].get("stage", "") == stg:
            ax.plot([x["epoch"] for x in h], [x["dev_f1"] for x in h], "o-", label=k)
    ax.set_title(stg or "(단계 없음)"); ax.set_xlabel("epoch"); ax.set_ylabel("dev F1"); ax.grid(alpha=.3); ax.legend(fontsize=7)
savefig("3_에폭별_dev")

## 4. 모델 비교 (test, 최종 점수)
막대는 test F1(micro)과 macro F1. 점선은 최고점에서 흔들림 std의 2배를 뺀 선이다. **이 선 위에 있는 모델들은 최고 모델과 통계적으로 구분하기 어렵다**고 본다.

In [ ]:
rows = []
for name, s in S_test.items():
    m = s["metrics"]; rows.append({"name": name, "member": s.get("member", ""), "stage": s.get("stage", ""),
        "test_F1": m["ALL"]["f1"], "P": m["ALL"]["precision"], "R": m["ALL"]["recall"], "macro_F1": m["MACRO_F1"],
        "dev_F1": S_dev.get(name, {}).get("metrics", {}).get("ALL", {}).get("f1")})
T = pd.DataFrame(rows).sort_values("test_F1")
T = T[~T.name.str.startswith(("D1_", "D2_"))]          # 데이터 실험은 위에서 따로 봄
if len(T):
    fig, ax = plt.subplots(figsize=(9, 0.45 * len(T) + 1.5))
    ax.barh(T.name, T.test_F1, label="test F1"); ax.barh(T.name, T.macro_F1, alpha=.45, label="macro F1")
    for y, v in enumerate(T.test_F1): ax.text(v, y, f" {v:.4f}", va="center", fontsize=8)
    if NOISE: ax.axvline(T.test_F1.max() - 2 * NOISE, ls="--", c="gray", label=f"최고점 − 2·std ({NOISE:.4f})")
    ax.set_xlim(max(0, T.macro_F1.min() - .05), 1); ax.legend(loc="lower right"); ax.set_title("모델 비교 (test)")
    savefig("4_모델비교")
T.sort_values("test_F1", ascending=False).to_csv(f"{RESULTS}/summary_table.csv", index=False); T.sort_values("test_F1", ascending=False)

## 5. 태그별·카테고리별 F1
모델마다 강한 태그·카테고리가 다르면 합쳤을 때 서로 보완할 여지가 있다는 신호다. 경제·산업 열은 2단계(금융) 출발 모델을 고르는 근거.

In [ ]:
names = list(T.name)[::-1]
if names:
    tagM = pd.DataFrame({n: {t: S_test[n]["metrics"][t]["f1"] for t in ev.TAGS} for n in names}).T
    cats = ["경제", "산업", "IT_과학", "정치", "사회일반", "연예"]
    catM = pd.DataFrame({n: {c: S_test[n]["by_category"].get(c, {}).get("f1") for c in cats} for n in names}).T
    fig, ax = plt.subplots(1, 2, figsize=(13, 0.45 * len(names) + 2))
    for a, M_, title in ((ax[0], tagM, "태그별 test F1"), (ax[1], catM, "카테고리별 test F1")):
        im = a.imshow(M_.values.astype(float), cmap="Greens", vmin=0, vmax=1, aspect="auto")
        a.set_xticks(range(M_.shape[1])); a.set_xticklabels(M_.columns); a.set_yticks(range(len(names))); a.set_yticklabels(names, fontsize=8)
        for i in range(M_.shape[0]):
            for j in range(M_.shape[1]):
                v = M_.values[i, j]
                if v == v and v is not None: a.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
        a.set_title(title)
    savefig("5_태그_카테고리")

## 6. 오답 분석 — 무엇을 틀리고, 서로 보완되는가

### 6-1. 오답 유형 (dev)
- **경계**: 태그는 맞는데 범위가 다름 (예: `금융통화위원회` vs `금융통화위원회는`)
- **유형**: 범위는 맞는데 태그가 다름
- **경계+유형**: 둘 다 다름 / **오탐**: 정답과 겹치지 않는 예측 / **미탐**: 예측과 겹치지 않는 정답
오답 목록은 `errors/{이름}_dev.jsonl` 에 문장·글자와 함께 있어 직접 읽어 볼 수 있다.

In [ ]:
E = pd.DataFrame({n: S_dev[n]["error_counts"] for n in names if n in S_dev}).T
if len(E):
    E.plot(kind="barh", stacked=True, figsize=(9, 0.45 * len(E) + 1.5), colormap="tab10"); plt.xlabel("오답 수 (dev)"); plt.title("오답 유형")
    savefig("6_오답유형")
E

### 6-2. 모델끼리 오답이 겹치는가 (dev로 판단)
**어떤 모델을 합칠지는 dev로 정한다.** test 오답을 보고 조합을 고르면 test 점수가 부풀려진다.

- **오라클 재현율**: 두 모델 중 하나라도 맞힌 정답의 비율. 두 모델이 서로 다른 것을 틀릴수록 높다. "완벽하게 합치면 이만큼 찾을 수 있다"는 상한.
- **오답 겹침률**: 둘 중 하나 이상이 틀린 정답 중, 둘 다 틀린 비율. 낮을수록 보완 효과가 크다.
- 결과를 글자 위치(span)로 저장했기 때문에, **토크나이저가 다른 모델끼리도** 바로 합칠 수 있다.

In [ ]:
DEV = f"{SPLIT}/dev.jsonl"
CAND = [n for n in names if os.path.exists(f"{RESULTS}/preds/{n}_dev.jsonl")][-5:]   # 상위 5개(필요하면 직접 지정)
print("비교 대상:", CAND)
if len(CAND) >= 2:
    dev_rows = load_jsonl(DEV)
    cmp = ev.compare(dev_rows, [ev.load_pred(f"{RESULTS}/preds/{n}_dev.jsonl") for n in CAND], CAND)
    k = len(CAND); O = np.full((k, k), np.nan)
    for i in range(k):
        O[i, i] = cmp["single"][CAND[i]]["recall"]
        for j in range(i + 1, k):
            O[i, j] = O[j, i] = cmp["pairs"][f"{CAND[i]} | {CAND[j]}"]["오라클_재현율"]
    fig, ax = plt.subplots(figsize=(1.2 * k + 3, 1.0 * k + 2))
    im = ax.imshow(O, cmap="Blues", vmin=np.nanmin(O) - .01, vmax=1)
    ax.set_xticks(range(k)); ax.set_xticklabels(CAND, rotation=45, ha="right", fontsize=8); ax.set_yticks(range(k)); ax.set_yticklabels(CAND, fontsize=8)
    for i in range(k):
        for j in range(k): ax.text(j, i, f"{O[i,j]:.3f}", ha="center", va="center", fontsize=8)
    ax.set_title("오라클 재현율 (대각선 = 단일 모델 재현율, dev)")
    savefig("6_오답보완_열지도")
    display(pd.DataFrame(cmp["pairs"]).T)
    display(pd.DataFrame({**{f"단일:{n}": v for n, v in cmp["single"].items()}, **cmp["ensembles"]}).T[["precision", "recall", "f1"]])
    json.dump(cmp, open(f"{RESULTS}/scores/compare_dev.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

### 6-3. 고른 조합을 test로 한 번만 확인
위 dev 결과로 조합(예: 다수결 3개)을 **먼저 정한 뒤**, 아래 `CHOSEN` 에 적고 test로 한 번만 채점한다.

In [ ]:
CHOSEN = []          # 예: ["M1_klue_roberta", "M1_koelectra", "M1_aihub_electra_1"]  ← dev로 정한 뒤 적기
if len(CHOSEN) >= 2:
    cmp_t = ev.compare(load_jsonl(f"{SPLIT}/test.jsonl"), [ev.load_pred(f"{RESULTS}/preds/{n}_test.jsonl") for n in CHOSEN], CHOSEN)
    json.dump(cmp_t, open(f"{RESULTS}/scores/compare_test_chosen.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    display(pd.DataFrame({**{f"단일:{n}": v for n, v in cmp_t["single"].items()}, **cmp_t["ensembles"]}).T[["precision", "recall", "f1"]])

## 7. 결론 적기 (팀 회의용 — 숫자가 없는 칸은 "측정 예정")
1. 학습 문장 수는 몇으로 정했나? 근거 그림: `1_학습량곡선.png`
2. 표본·시드 흔들림 std는 얼마였나? 근거: `2_표본흔들림.png`
3. 에폭·학습률은 어떻게 정했나? 근거: `3_에폭별_dev.png`
4. 최고 모델과 흔들림 범위 안에 있는 모델은? 근거: `4_모델비교.png`
5. 태그·카테고리별 강점은? 경제·산업에서 가장 좋은 모델은? 근거: `5_태그_카테고리.png`
6. 오답 유형 중 가장 큰 것은? 모델끼리 보완되나(오라클 재현율), 앙상블은 단일 모델보다 나았나? 근거: `6_*.png`